# Day 2 - Preparing Dataset to Used for Batching

In [28]:
import os
import json
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from openai import OpenAI
from litellm import completion
from tqdm.notebook import tqdm
from IPython.display import display, Markdown
from pricer.evaluator import evaluate

In [29]:
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [30]:
openai = OpenAI(api_key=OPENAI_API_KEY)

In [31]:
ADMR = "ADMR.JK"
PANI = "PANI.JK"
PTRO = "PTRO.JK"
BUMI = "BUMI.JK"
HORIZON = 30
MODEL = "gpt-4.1-nano"

In [40]:
df = yf.Tickers(f"{ADMR} {PANI} {PTRO} {BUMI}").history(period="max")

[*********************100%***********************]  4 of 4 completed


In [41]:
df

Price        Close                             Dividends                  \
Ticker     ADMR.JK     BUMI.JK PANI.JK PTRO.JK   ADMR.JK BUMI.JK PANI.JK   
Date                                                                       
2001-06-07     NaN   42.833714     NaN     NaN       NaN     0.0     NaN   
2001-06-08     NaN   42.833714     NaN     NaN       NaN     0.0     NaN   
2001-06-11     NaN   47.117092     NaN     NaN       NaN     0.0     NaN   
2001-06-12     NaN   47.117092     NaN     NaN       NaN     0.0     NaN   
2001-06-13     NaN   42.833714     NaN     NaN       NaN     0.0     NaN   
...            ...         ...     ...     ...       ...     ...     ...   
2026-04-20  1905.0  242.000000  8750.0  6075.0       0.0     0.0     0.0   
2026-04-21  1865.0  240.000000  9425.0  6300.0       0.0     0.0     0.0   
2026-04-22  1855.0  240.000000  9225.0  6250.0       0.0     0.0     0.0   
2026-04-23  1845.0  230.000000  9050.0  6175.0       0.0     0.0     0.0   
2026-04-24  1880.0  216.000000  8500.0  5600.0       0.0     0.0     0.0   

Price                 High              ...    Open         Stock Splits  \
Ticker     PTRO.JK ADMR.JK     BUMI.JK  ... PANI.JK PTRO.JK      ADMR.JK   
Date                                    ...                                
2001-06-07     NaN     NaN   42.833714  ...     NaN     NaN          NaN   
2001-06-08     NaN     NaN   42.833714  ...     NaN     NaN          NaN   
2001-06-11     NaN     NaN   51.400464  ...     NaN     NaN          NaN   
2001-06-12     NaN     NaN   51.400464  ...     NaN     NaN          NaN   
2001-06-13     NaN     NaN   47.117085  ...     NaN     NaN          NaN   
...            ...     ...         ...  ...     ...     ...          ...   
2026-04-20     0.0  1940.0  250.000000  ...  9000.0  6250.0          0.0   
2026-04-21     0.0  1900.0  244.000000  ...  8775.0  6025.0          0.0   
2026-04-22     0.0  1885.0  246.000000  ...  9475.0  6350.0          0.0   
2026-04-23     0.0  1880.0  242.000000  ...  9225.0  6275.0          0.0   
2026-04-24     0.0  1925.0  236.000000  ...  9050.0  6225.0          0.0   

Price                                   Volume                          \
Ticker     BUMI.JK PANI.JK PTRO.JK     ADMR.JK     BUMI.JK     PANI.JK   
Date                                                                     
2001-06-07     0.0     NaN     NaN         NaN     3869000         NaN   
2001-06-08     0.0     NaN     NaN         NaN     6038500         NaN   
2001-06-11     0.0     NaN     NaN         NaN     8006000         NaN   
2001-06-12     0.0     NaN     NaN         NaN      537000         NaN   
2001-06-13     0.0     NaN     NaN         NaN     1341500         NaN   
...            ...     ...     ...         ...         ...         ...   
2026-04-20     0.0     0.0     0.0  19839200.0  2320011600   6120200.0   
2026-04-21     0.0     0.0     0.0  36164700.0  2190621300  11543800.0   
2026-04-22     0.0     0.0     0.0  25625300.0  1883526900   6620900.0   
2026-04-23     0.0     0.0     0.0  25486300.0  3490960700   4322000.0   
2026-04-24     0.0     0.0     0.0  48337900.0  4442692000   8074300.0   

Price                    
Ticker          PTRO.JK  
Date                     
2001-06-07          NaN  
2001-06-08          NaN  
2001-06-11          NaN  
2001-06-12          NaN  
2001-06-13          NaN  
...                 ...  
2026-04-20   50766900.0  
2026-04-21   98619000.0  
2026-04-22   65449000.0  
2026-04-23   62695100.0  
2026-04-24  105694400.0  

[6177 rows x 28 columns]

In [42]:
close_df = df["Close"]

# MA and momentum features
ma_200 = close_df.rolling(window=200).mean()
ma_150 = close_df.rolling(window=150).mean()
ma_50  = close_df.rolling(window=50).mean()
momentum_10 = close_df.pct_change(periods=10)

# RSI features
delta = close_df.diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
rsi = 100 - (100 / (1 + rs))

In [43]:
df = pd.concat([
    df,
    ma_200.rename(columns=lambda x: ("MA 200", x)),
    ma_150.rename(columns=lambda x: ("MA 150", x)),
    ma_50.rename(columns=lambda x: ("MA 50", x)),
    momentum_10.rename(columns=lambda x: ("Momentum 10", x)),
    rsi.rename(columns=lambda x: ("RSI 14", x))
], axis=1)

In [44]:
df.columns.names = ["Price", "Ticker"]

In [45]:
tickers = [ADMR, PANI, PTRO, BUMI]

window_size = 30
forecast_horizon = 30
idx = 0

dataset = []

for ticker in tqdm(tickers):
    stock_df = df.xs(ticker, axis=1, level=1).copy()

    stock_df = stock_df.reset_index()

    stock_df = stock_df.rename(columns={stock_df.columns[0]: "Date"})

    stock_df = stock_df[["Date", "Close", "Volume", "MA 200", "MA 150", "MA 50", "Momentum 10", "RSI 14"]]

    stock_df = stock_df.dropna().reset_index(drop=True)

    for i in range(len(stock_df) - window_size - forecast_horizon):
        past_30_days = stock_df.iloc[i:i+window_size]

        target_close = stock_df.iloc[
            i + window_size + forecast_horizon - 1
        ]["Close"]
        return_percent = ((target_close - past_30_days.iloc[-1]["Close"]) / past_30_days.iloc[-1]["Close"]) * 100

        rows_text = []

        for _, row in past_30_days.iterrows():
            rows_text.append(
                f"Date: {row['Date']}, "
                f"Close: {row['Close']:.2f}, "
                f"Volume: {int(row['Volume'])}, "
                f"MA-200: {row['MA 200']:.2f}, "
                f"MA-150: {row['MA 150']:.2f}, "
                f"MA-50: {row['MA 50']:.2f}, "
                f"Momentum-10: {row['Momentum 10']:.2f}, "
                f"RSI-14: {row['RSI 14']:.2f}"
            )

        price_history = "\n".join(rows_text)

        dataset.append({
            "Id": idx,
            "Ticker": ticker,
            "Start Date": past_30_days.iloc[0]["Date"],
            "End Date": past_30_days.iloc[-1]["Date"],
            "Last Price": past_30_days.iloc[-1]["Close"],
            "Price History": price_history,
            "Future Price": f"{target_close:.2f}",
            "Return %": f"{return_percent:.2f}"
        })

        idx += 1

processed_df = pd.DataFrame(dataset)

  0%|          | 0/4 [00:00<?, ?it/s]

In [46]:
print(f"Sample instruction and completion: {len(processed_df)}\n")
print(processed_df.iloc[0]["Price History"])
print(f"\nExpected completion: {processed_df.iloc[0]['Future Price']}")

Sample instruction and completion: 13336

Date: 2022-10-27 00:00:00, Close: 1667.96, Volume: 39471900, MA-200: 1657.68, MA-150: 1862.11, MA-50: 1675.29, Momentum-10: 0.01, RSI-14: 31.43
Date: 2022-10-28 00:00:00, Close: 1644.13, Volume: 28087800, MA-200: 1665.26, MA-150: 1862.26, MA-50: 1674.63, Momentum-10: 0.01, RSI-14: 33.85
Date: 2022-10-31 00:00:00, Close: 1648.89, Volume: 28997300, MA-200: 1672.64, MA-150: 1861.63, MA-50: 1676.25, Momentum-10: -0.00, RSI-14: 41.82
Date: 2022-11-01 00:00:00, Close: 1606.00, Volume: 27783400, MA-200: 1679.50, MA-150: 1860.64, MA-50: 1676.25, Momentum-10: -0.03, RSI-14: 39.66
Date: 2022-11-02 00:00:00, Close: 1667.96, Volume: 43630200, MA-200: 1686.39, MA-150: 1860.01, MA-50: 1678.34, Momentum-10: -0.01, RSI-14: 51.43
Date: 2022-11-03 00:00:00, Close: 1663.19, Volume: 28675500, MA-200: 1692.90, MA-150: 1858.61, MA-50: 1680.63, Momentum-10: -0.03, RSI-14: 55.38
Date: 2022-11-04 00:00:00, Close: 1682.25, Volume: 21295200, MA-200: 1699.05, MA-150: 1857

In [47]:
processed_df.to_csv("dfs/processed_stock_data.csv", index=False)